# Module 5 – Embeddings, Indexing & Retrieval

## 📍 Where We Are in the Pipeline

```
Document → Extract → Chunk → EMBED → INDEX → RETRIEVE → Generate
                     ✅ M4    🔵 NOW  🔵 NOW  🔵 NOW
```

**This module covers THREE critical pipeline stages:**
1. **🧮 EMBED** – Convert text chunks to 3072-dimensional vectors
2. **📦 INDEX** – Store vectors in Azure AI Search
3. **🔎 RETRIEVE** – Find relevant chunks for user queries

---

## Learning Outcomes

By the end of this module, you will be able to:
- Generate embeddings using `text-embedding-3-large`
- Design index schemas for RAG workloads with vector fields
- Create and populate an Azure AI Search index (Push model)
- Implement text, vector, and hybrid search
- Configure semantic ranking for improved relevance
- Select the right retrieval pattern for different use cases
- **Use Agentic Retrieval for complex multi-part questions (Preview)**

---

## ⏱️ Estimated Time: ~3 hours

| Section | Time |
|---------|------|
| Part 0: Setup & Load Chunks | 10 min |
| Part 1: Embeddings | 30 min |
| Part 2: Index Creation | 30 min |
| Part 3: Search Modes | 45 min |
| Part 4: Retrieval Patterns | 45 min |
| Part 5: Agentic Retrieval (Preview) | 30 min |

---

# Part 0: Setup & Load Chunks from Module 4

First, let's load our environment and the chunks we created in Module 4.

In [ ]:
# Cell 0.1: Install/verify dependencies
import sys
!{sys.executable} -m pip install -q azure-search-documents==11.6.0 openai python-dotenv tqdm azure-identity

In [ ]:
# Cell 0.2: Load environment variables
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Find and load .env from project root
project_root = Path.cwd()
while project_root != project_root.parent:
    if (project_root / ".env").exists():
        load_dotenv(project_root / ".env")
        print(f"✅ Loaded .env from {project_root}")
        break
    project_root = project_root.parent
else:
    raise FileNotFoundError("❌ .env file not found. Run Module 0 setup first.")

# Verify required environment variables
required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_DEPLOYMENT_EMBEDDING",
    "AZURE_SEARCH_ENDPOINT",
    "AZURE_SEARCH_API_KEY",
]

missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise ValueError(f"❌ Missing environment variables: {missing}")

print("✅ All required environment variables loaded")
print(f"   - OpenAI Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')[:50]}...")
print(f"   - Search Endpoint: {os.getenv('AZURE_SEARCH_ENDPOINT')[:50]}...")

In [ ]:
# Cell 0.3: Load chunks from Module 4
chunks_path = project_root / "modules" / "module-4-chunking" / "output" / "hybrid_chunks.json"

if not chunks_path.exists():
    raise FileNotFoundError(f"❌ Chunks file not found: {chunks_path}\n   Please complete Module 4 first.")

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ Loaded {len(chunks)} chunks from Module 4")

# Analyze chunk distribution
content_types = {}
for chunk in chunks:
    ct = chunk.get("content_type", "unknown")
    content_types[ct] = content_types.get(ct, 0) + 1

print("\n📊 Chunk Distribution:")
for ct, count in sorted(content_types.items(), key=lambda x: -x[1]):
    print(f"   {ct}: {count}")

In [ ]:
# Cell 0.4: Inspect a few chunks
print("📄 Sample Chunks:\n")

# Show one of each type
shown_types = set()
for chunk in chunks:
    ct = chunk.get("content_type", "unknown")
    if ct not in shown_types:
        shown_types.add(ct)
        print(f"--- {ct.upper()} (id: {chunk['id']}) ---")
        content = chunk['content'][:300] + "..." if len(chunk['content']) > 300 else chunk['content']
        print(content)
        print(f"\nMetadata: {chunk.get('metadata', {})}")
        print("\n")
    if len(shown_types) >= 3:
        break

---

# Part 1: Embeddings

## What are Embeddings?

Embeddings are **dense vector representations** of text that capture semantic meaning:
- Similar concepts have vectors that are close together
- `text-embedding-3-large` produces **3072-dimensional** vectors
- Enable **semantic search** beyond keyword matching

```
"electric motor"  →  [0.023, -0.156, 0.089, ..., 0.042]  (3072 floats)
"DC motor"        →  [0.019, -0.148, 0.092, ..., 0.038]  (similar!)
"cooking recipe"  →  [-0.234, 0.078, -0.156, ..., -0.089]  (very different)
```

## Lab 1.1: Initialize OpenAI Client

In [ ]:
# Cell 1.1: Initialize Azure OpenAI client for embeddings
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Use Entra ID authentication (keys are disabled on this resource)
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_ad_token_provider=token_provider,
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
)

EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_EMBEDDING", "text-embedding-3-large")
EMBEDDING_DIMENSIONS = 3072

print(f"✅ OpenAI client initialized (using Entra ID)")
print(f"   - Embedding model: {EMBEDDING_MODEL}")
print(f"   - Dimensions: {EMBEDDING_DIMENSIONS}")

## Lab 1.2: Generate a Single Embedding

In [ ]:
# Cell 1.2: Generate embedding for a single text
def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> list[float]:
    """
    Generate embedding for a single text.
    
    Args:
        text: Input text (max ~8191 tokens)
        model: Embedding model deployment name
        
    Returns:
        List of floats (3072 dimensions)
    """
    # Clean and truncate text if needed (rough estimate: 4 chars per token)
    max_chars = 8000 * 4  # ~32000 chars
    if len(text) > max_chars:
        text = text[:max_chars]
    
    response = openai_client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

# Test with a simple example
test_text = "Kirchhoff's voltage law states that the sum of voltages around a closed loop equals zero."
test_embedding = get_embedding(test_text)

print(f"✅ Generated embedding for test text")
print(f"   - Input length: {len(test_text)} characters")
print(f"   - Output dimensions: {len(test_embedding)}")
print(f"   - First 5 values: {test_embedding[:5]}")

## Lab 1.3: Semantic Similarity Demo

Let's verify that embeddings capture semantic similarity:

In [ ]:
# Cell 1.3: Demonstrate semantic similarity with cosine distance
import numpy as np

def cosine_similarity(v1: list[float], v2: list[float]) -> float:
    """Calculate cosine similarity between two vectors."""
    a = np.array(v1)
    b = np.array(v2)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Test sentences
sentences = [
    "Kirchhoff's voltage law states that the sum of voltages in a closed loop is zero.",
    "KVL says voltages around any closed circuit path add up to zero.",  # Same concept, different words
    "The DC motor converts electrical energy to mechanical energy.",      # Different concept
    "I love eating pizza on Friday nights."                               # Completely unrelated
]

print("📊 Semantic Similarity Matrix\n")
print(f"{'':>5}", end="")
for i in range(len(sentences)):
    print(f"  S{i+1}  ", end="")
print("\n")

embeddings = [get_embedding(s) for s in sentences]

for i, emb_i in enumerate(embeddings):
    print(f"S{i+1}  ", end="")
    for j, emb_j in enumerate(embeddings):
        sim = cosine_similarity(emb_i, emb_j)
        print(f" {sim:.3f} ", end="")
    print()

print("\n📝 Sentences:")
for i, s in enumerate(sentences):
    print(f"S{i+1}: {s[:60]}..." if len(s) > 60 else f"S{i+1}: {s}")

print("\n💡 Observation: S1 and S2 (same concept) should have highest similarity (~0.9+)")

## Lab 1.4: Batch Embedding Generation

For efficiency, we process multiple texts in batches:

In [ ]:
# Cell 1.4: Batch embedding function
from tqdm import tqdm
import time

def get_embeddings_batch(
    texts: list[str], 
    model: str = EMBEDDING_MODEL, 
    batch_size: int = 16,
    show_progress: bool = True
) -> list[list[float]]:
    """
    Generate embeddings for multiple texts in batches.
    
    Args:
        texts: List of input texts
        model: Embedding model deployment name
        batch_size: Number of texts per API call (max ~16 recommended)
        show_progress: Show progress bar
        
    Returns:
        List of embedding vectors
    """
    all_embeddings = []
    max_chars = 8000 * 4  # Token limit safety
    
    # Process in batches
    batches = [texts[i:i+batch_size] for i in range(0, len(texts), batch_size)]
    
    iterator = tqdm(batches, desc="Generating embeddings") if show_progress else batches
    
    for batch in iterator:
        # Truncate long texts
        batch_cleaned = [t[:max_chars] if len(t) > max_chars else t for t in batch]
        
        try:
            response = openai_client.embeddings.create(
                input=batch_cleaned,
                model=model
            )
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
        except Exception as e:
            print(f"❌ Error in batch: {e}")
            # Add empty embeddings for failed batch (handle gracefully)
            all_embeddings.extend([[0.0] * EMBEDDING_DIMENSIONS] * len(batch))
        
        # Rate limiting - be nice to the API
        time.sleep(0.1)
    
    return all_embeddings

print("✅ Batch embedding function defined")

## Lab 1.5: Generate Embeddings for All Chunks

Now let's embed all our chunks from Module 4. This may take a few minutes.

In [ ]:
# Cell 1.5: Generate embeddings for all chunks

# Extract text content from chunks
chunk_texts = [chunk["content"] for chunk in chunks]

print(f"📊 Embedding {len(chunk_texts)} chunks...")
print(f"   - Estimated time: ~{len(chunk_texts) // 16 * 2} seconds (varies by load)\n")

start_time = time.time()
embeddings = get_embeddings_batch(chunk_texts, batch_size=16)
elapsed = time.time() - start_time

print(f"\n✅ Generated {len(embeddings)} embeddings in {elapsed:.1f}s")
print(f"   - Rate: {len(embeddings)/elapsed:.1f} embeddings/sec")
print(f"   - Each embedding: {len(embeddings[0])} dimensions")

In [ ]:
# Cell 1.6: Attach embeddings to chunks

# Create enriched chunks with embeddings
enriched_chunks = []
for i, chunk in enumerate(chunks):
    enriched_chunk = chunk.copy()
    enriched_chunk["embedding"] = embeddings[i]
    enriched_chunks.append(enriched_chunk)

print(f"✅ Created {len(enriched_chunks)} enriched chunks with embeddings")

# Verify
sample = enriched_chunks[0]
print(f"\n📄 Sample enriched chunk:")
print(f"   - id: {sample['id']}")
print(f"   - content_type: {sample.get('content_type', 'unknown')}")
print(f"   - content length: {len(sample['content'])} chars")
print(f"   - embedding dimensions: {len(sample['embedding'])}")

---

# Part 2: Azure AI Search Index

## Azure AI Search Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Azure AI Search                          │
├─────────────────────────────────────────────────────────────┤
│  ┌──────────┐   ┌──────────┐   ┌──────────┐                │
│  │  INDEX   │   │  INDEX   │   │  INDEX   │   ...          │
│  │(schema)  │   │(schema)  │   │(schema)  │                │
│  └────┬─────┘   └──────────┘   └──────────┘                │
│       │                                                     │
│       ▼                                                     │
│  ┌──────────────────────────────────────┐                  │
│  │           DOCUMENTS                   │                  │
│  │  ┌────┐ ┌────┐ ┌────┐ ┌────┐ ...    │                  │
│  │  │doc1│ │doc2│ │doc3│ │doc4│         │                  │
│  │  └────┘ └────┘ └────┘ └────┘         │                  │
│  └──────────────────────────────────────┘                  │
└─────────────────────────────────────────────────────────────┘
```

**Key Concepts:**
- **Index**: Schema definition (like a database table)
- **Document**: Individual item in the index (like a row)
- **Field**: Attribute of a document (like a column)
- **Vector Field**: Special field type for semantic search

## Lab 2.1: Initialize Search Client

In [ ]:
# Cell 2.1: Initialize Azure AI Search clients
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SearchableField,
    SimpleField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
)

# Configuration
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME", "rag-workshop-index")

# Create clients - try Entra ID first, fall back to API key if available
if SEARCH_API_KEY:
    search_credential = AzureKeyCredential(SEARCH_API_KEY)
    print("   - Auth: API Key")
else:
    search_credential = credential  # Use Entra ID credential from earlier
    print("   - Auth: Entra ID (DefaultAzureCredential)")

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=search_credential)

print(f"✅ Search clients initialized")
print(f"   - Endpoint: {SEARCH_ENDPOINT}")
print(f"   - Index name: {INDEX_NAME}")

## Lab 2.2: Design the Index Schema

A well-designed schema is critical for RAG performance:

| Field | Type | Purpose |
|-------|------|----------|
| `id` | string | Unique identifier (key) |
| `content` | string | Searchable text content |
| `content_type` | string | Type (text, table, figure) for filtering |
| `embedding` | vector(3072) | Semantic search vector |
| `strategy` | string | Chunking strategy used |
| `metadata` | string | JSON metadata (flexible) |

In [ ]:
# Cell 2.2: Define the index schema

# Vector search configuration
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="hnsw-config",
            parameters={
                "m": 4,          # Number of bi-directional links (default: 4)
                "efConstruction": 400,  # Size of dynamic list during indexing
                "efSearch": 500,        # Size of dynamic list during search
                "metric": "cosine"      # Distance metric
            }
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="vector-profile",
            algorithm_configuration_name="hnsw-config"
        )
    ]
)

# Semantic search configuration (for L2 reranking)
semantic_config = SemanticConfiguration(
    name="semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")],
    )
)

semantic_search = SemanticSearch(configurations=[semantic_config])

# Define fields
fields = [
    # Key field (required)
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True
    ),
    # Content field (searchable)
    SearchableField(
        name="content",
        type=SearchFieldDataType.String,
        searchable=True,
        analyzer_name="en.microsoft"  # English language analyzer
    ),
    # Content type (for filtering by chunk type)
    SimpleField(
        name="content_type",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True
    ),
    # Strategy field
    SimpleField(
        name="strategy",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True
    ),
    # Vector embedding field
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIMENSIONS,
        vector_search_profile_name="vector-profile"
    ),
    # Metadata as JSON string
    SimpleField(
        name="metadata",
        type=SearchFieldDataType.String,
        filterable=False
    ),
]

print("✅ Index schema defined")
print(f"\n📋 Fields:")
for f in fields:
    print(f"   - {f.name}: {f.type}")

## Lab 2.3: Create the Index

In [ ]:
# Cell 2.3: Create or update the index

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

# Create or update
try:
    result = index_client.create_or_update_index(index)
    print(f"✅ Index '{result.name}' created/updated successfully")
except Exception as e:
    print(f"❌ Error creating index: {e}")
    raise

## Lab 2.4: Upload Documents (Push Model)

Azure AI Search supports two ingestion patterns:
- **Push Model**: Application uploads documents directly via SDK
- **Pull Model**: Indexer pulls from data source (Blob, SQL, etc.)

For RAG with pre-computed embeddings, **Push** is typically better.

In [ ]:
# Cell 2.4: Prepare documents for upload

def prepare_document(chunk: dict) -> dict:
    """
    Convert a chunk to a search document.
    
    Args:
        chunk: Enriched chunk with embedding
        
    Returns:
        Document dict ready for indexing
    """
    return {
        "id": chunk["id"],
        "content": chunk["content"],
        "content_type": chunk.get("content_type", "text"),
        "strategy": chunk.get("strategy", "unknown"),
        "embedding": chunk["embedding"],
        "metadata": json.dumps(chunk.get("metadata", {}))
    }

# Prepare all documents
documents = [prepare_document(chunk) for chunk in enriched_chunks]

print(f"✅ Prepared {len(documents)} documents for upload")
print(f"\n📄 Sample document:")
sample_doc = documents[0].copy()
sample_doc["embedding"] = f"[{len(sample_doc['embedding'])} floats]"  # Truncate for display
sample_doc["content"] = sample_doc["content"][:100] + "..."
for k, v in sample_doc.items():
    print(f"   {k}: {v}")

In [ ]:
# Cell 2.5: Upload documents in batches

# Create search client for document operations
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=search_credential
)

# Upload in batches of 100 (recommended max)
batch_size = 100
total_uploaded = 0
total_failed = 0

print(f"📤 Uploading {len(documents)} documents in batches of {batch_size}...\n")

for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    try:
        result = search_client.upload_documents(documents=batch)
        succeeded = sum(1 for r in result if r.succeeded)
        failed = len(batch) - succeeded
        total_uploaded += succeeded
        total_failed += failed
        print(f"   Batch {i//batch_size + 1}: {succeeded} succeeded, {failed} failed")
    except Exception as e:
        print(f"   Batch {i//batch_size + 1}: ❌ Error - {e}")
        total_failed += len(batch)

print(f"\n✅ Upload complete: {total_uploaded} succeeded, {total_failed} failed")

In [ ]:
# Cell 2.6: Verify index population
import time

# Wait a moment for index to update
time.sleep(2)

# Get document count
results = search_client.search(search_text="*", include_total_count=True)
total_count = results.get_count()

print(f"✅ Index '{INDEX_NAME}' now contains {total_count} documents")

# Get facets by content_type
facet_results = search_client.search(
    search_text="*",
    facets=["content_type"],
    top=0
)
facets = facet_results.get_facets()

if facets and "content_type" in facets:
    print(f"\n📊 Content type distribution:")
    for facet in facets["content_type"]:
        print(f"   - {facet['value']}: {facet['count']}")

---

# Part 3: Search Modes

Azure AI Search supports multiple search modes:

| Mode | How it Works | Best For |
|------|-------------|----------|
| **Text (BM25)** | Keyword matching + TF-IDF | Exact terms, codes |
| **Vector** | Cosine similarity on embeddings | Semantic meaning |
| **Hybrid** | BM25 + Vector with RRF fusion | General RAG |
| **Semantic** | Hybrid + L2 neural reranking | Production RAG |

## Lab 3.1: Text Search (BM25)

In [ ]:
# Cell 3.1: Text-only search (BM25)
from azure.search.documents.models import QueryType

def search_text(query: str, top: int = 5, filter: str = None) -> list[dict]:
    """
    Perform text-only (BM25) search.
    
    Args:
        query: Search query text
        top: Number of results
        filter: OData filter expression
        
    Returns:
        List of search results with score
    """
    results = search_client.search(
        search_text=query,
        query_type=QueryType.SIMPLE,
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "strategy"]
    )
    
    return [
        {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "score": r["@search.score"]
        }
        for r in results
    ]

# Test text search
query = "Kirchhoff's voltage law"
text_results = search_text(query, top=3)

print(f"🔍 Text Search (BM25): '{query}'\n")
for i, r in enumerate(text_results, 1):
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}")
    print(f"   {r['content'][:150]}...\n")

## Lab 3.2: Vector Search (Semantic)

In [ ]:
# Cell 3.2: Vector-only search
from azure.search.documents.models import VectorizedQuery

def search_vector(query: str, top: int = 5, filter: str = None) -> list[dict]:
    """
    Perform vector-only (semantic) search.
    
    Args:
        query: Search query text (will be embedded)
        top: Number of results
        filter: OData filter expression
        
    Returns:
        List of search results with score
    """
    # Get embedding for query
    query_embedding = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=top,
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=None,  # No text search
        vector_queries=[vector_query],
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "strategy"]
    )
    
    return [
        {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "score": r["@search.score"]
        }
        for r in results
    ]

# Test vector search with semantic query
query = "What is the law about voltages summing to zero in circuits?"
vector_results = search_vector(query, top=3)

print(f"🔍 Vector Search (Semantic): '{query}'\n")
for i, r in enumerate(vector_results, 1):
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}")
    print(f"   {r['content'][:150]}...\n")

## Lab 3.3: Hybrid Search (Text + Vector)

**Hybrid search** combines BM25 and vector search using **Reciprocal Rank Fusion (RRF)**:

```
RRF_score = 1/(k + rank_bm25) + 1/(k + rank_vector)
```

This gives the best of both worlds:
- Exact keyword matching from BM25
- Semantic understanding from vectors

In [ ]:
# Cell 3.3: Hybrid search (text + vector)

def search_hybrid(query: str, top: int = 5, filter: str = None) -> list[dict]:
    """
    Perform hybrid search (BM25 + vector with RRF fusion).
    
    Args:
        query: Search query text
        top: Number of results
        filter: OData filter expression
        
    Returns:
        List of search results with score
    """
    # Get embedding for query
    query_embedding = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=50,  # Over-fetch for RRF
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=query,        # Text search
        vector_queries=[vector_query],  # Vector search
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "strategy"]
    )
    
    return [
        {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "score": r["@search.score"]
        }
        for r in results
    ]

# Test hybrid search
query = "KVL Kirchhoff voltage law"
hybrid_results = search_hybrid(query, top=3)

print(f"🔍 Hybrid Search (BM25 + Vector): '{query}'\n")
for i, r in enumerate(hybrid_results, 1):
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}")
    print(f"   {r['content'][:150]}...\n")

## Lab 3.4: Semantic Ranking (L2 Reranker)

### What is Semantic Ranking?

**Semantic ranking** (also called **L2 reranking**) is a two-stage retrieval process:

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Stage 1 (L1): Hybrid Search                                            │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Query → BM25 + Vector → RRF Fusion → Top 50 candidates         │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                              ↓                                          │
│  Stage 2 (L2): Semantic Reranker                                        │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Transformer model scores each candidate for relevance          │   │
│  │  Cross-encoder compares (query, document) pairs deeply          │   │
│  │  Returns reranker_score (0-4 scale) + final top K               │   │
│  └─────────────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────────────┘
```

### Why Two Stages?

| Stage | Method | Speed | Quality | Purpose |
|-------|--------|-------|---------|---------|
| **L1** | BM25 + Vector | Fast (ms) | Good | Narrow down from millions to ~50 |
| **L2** | Cross-encoder | Slower | Excellent | Rerank 50 candidates precisely |

The L1 stage uses **bi-encoders** (separate embeddings for query and docs) which are fast but less accurate. The L2 stage uses a **cross-encoder** that processes query+document together for deeper semantic understanding.

### Reranker Score (0-4 Scale)

The semantic reranker returns a score from **0 to 4**:

| Score | Meaning |
|-------|---------|
| 0 | Not relevant at all |
| 1 | Slightly relevant |
| 2 | Moderately relevant |
| 3 | Highly relevant |
| **4** | Perfect match |

> 💡 **Tip**: Filter results with `reranker_score >= 2` for quality answers.

### Additional Features

**Extractive Answers**: The reranker can extract specific text spans that directly answer the question.

**Captions**: Highlighted passages showing why a document matched.

In [ ]:
# Cell 3.4: Hybrid + Semantic ranking
from azure.search.documents.models import QueryType, QueryCaptionType, QueryAnswerType

def search_semantic(
    query: str, 
    top: int = 5, 
    filter: str = None,
    include_answers: bool = True
) -> dict:
    """
    Perform hybrid search with semantic ranking.
    
    Args:
        query: Search query text
        top: Number of results
        filter: OData filter expression
        include_answers: Extract semantic answers
        
    Returns:
        Dict with results and optional answers
    """
    # Get embedding for query
    query_embedding = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=50,
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="semantic-config",
        query_caption=QueryCaptionType.EXTRACTIVE,
        query_answer=QueryAnswerType.EXTRACTIVE if include_answers else None,
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "strategy"]
    )
    
    # Extract results
    output = {
        "results": [],
        "answers": []
    }
    
    # Get answers (if available)
    try:
        answers = results.get_answers()
        if answers:
            output["answers"] = [
                {
                    "text": a.text,
                    "score": a.score
                }
                for a in answers
            ]
    except:
        pass
    
    # Get documents
    for r in results:
        doc = {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "score": r["@search.score"],
            "reranker_score": r.get("@search.reranker_score", None)
        }
        
        # Get captions
        captions = r.get("@search.captions", [])
        if captions:
            doc["caption"] = captions[0].text if hasattr(captions[0], "text") else str(captions[0])
        
        output["results"].append(doc)
    
    return output

# Test semantic search
query = "How does a DC motor convert electrical energy to mechanical energy?"
semantic_results = search_semantic(query, top=3)

print(f"🔍 Semantic Search: '{query}'\n")

# Show answers (if any)
if semantic_results["answers"]:
    print("📝 Extracted Answers:")
    for a in semantic_results["answers"]:
        print(f"   Score {a['score']:.2f}: {a['text']}")
    print()

# Show documents
print("📄 Documents:")
for i, r in enumerate(semantic_results["results"], 1):
    reranker_str = f", Reranker: {r['reranker_score']:.2f}" if r['reranker_score'] else ""
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}{reranker_str}")
    if r.get("caption"):
        print(f"   Caption: {r['caption'][:100]}...")
    print(f"   Content: {r['content'][:100]}...\n")

## Lab 3.5: Compare Search Modes

Let's compare all search modes side by side:

In [ ]:
# Cell 3.5: Side-by-side comparison

def compare_search_modes(query: str, top: int = 3):
    """Compare different search modes for the same query."""
    print(f"="*80)
    print(f"Query: '{query}'")
    print(f"="*80)
    
    # Text search
    text_results = search_text(query, top)
    print(f"\n📖 TEXT (BM25):")
    for i, r in enumerate(text_results, 1):
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f})")
    
    # Vector search
    vector_results = search_vector(query, top)
    print(f"\n🧮 VECTOR (Cosine):")
    for i, r in enumerate(vector_results, 1):
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f})")
    
    # Hybrid search
    hybrid_results = search_hybrid(query, top)
    print(f"\n🔀 HYBRID (RRF):")
    for i, r in enumerate(hybrid_results, 1):
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f})")
    
    # Semantic search
    semantic_results = search_semantic(query, top, include_answers=False)
    print(f"\n🧠 SEMANTIC (L2 Reranker):")
    for i, r in enumerate(semantic_results["results"], 1):
        reranker = f", reranker: {r['reranker_score']:.2f}" if r['reranker_score'] else ""
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f}{reranker})")

# Test with different query types
print("\n" + "🔬 TEST 1: Technical term")
compare_search_modes("KVL circuit analysis")

print("\n" + "🔬 TEST 2: Natural language question")
compare_search_modes("How do I calculate power in an AC circuit?")

print("\n" + "🔬 TEST 3: Abbreviation + full term")
compare_search_modes("EMF electromotive force generator")

---

# Part 4: Retrieval Patterns for RAG

Different RAG scenarios require different retrieval strategies:

| Pattern | Use Case |
|---------|----------|
| **Multi-Retriever** | Mixed content (text + tables + figures) |
| **Filtered Retrieval** | Content-type specific queries |
| **Multimodal** | Questions about diagrams/charts |

## Lab 4.1: Multi-Retriever Pattern

For technical documents, retrieve from each content type separately and merge:

In [ ]:
# Cell 4.1: Multi-retriever with content-type awareness

def multi_retriever(
    query: str,
    top_per_type: int = 2,
    content_types: list[str] = ["text", "table", "figure"]
) -> dict:
    """
    Retrieve from each content type separately.
    
    This pattern ensures tables and figures aren't drowned out
    by text chunks in the results.
    
    Args:
        query: Search query
        top_per_type: Results per content type
        content_types: Types to query
        
    Returns:
        Dict with results by content type
    """
    results = {}
    
    for ct in content_types:
        filter_expr = f"content_type eq '{ct}'"
        type_results = search_hybrid(query, top=top_per_type, filter=filter_expr)
        results[ct] = type_results
    
    return results

# Test multi-retriever
query = "types of winding in DC machines"
multi_results = multi_retriever(query, top_per_type=2)

print(f"🔍 Multi-Retriever: '{query}'\n")

for content_type, results in multi_results.items():
    print(f"📁 {content_type.upper()} ({len(results)} results):")
    for r in results:
        print(f"   - {r['id']}: {r['content'][:80]}...")
    print()

## Lab 4.2: Filtered Retrieval

Use filters to narrow search based on user intent:

In [ ]:
# Cell 4.2: Intent-based filtered retrieval

def detect_intent(query: str) -> str:
    """
    Simple intent detection for filtering.
    In production, use an LLM for this.
    """
    query_lower = query.lower()
    
    # Check for table indicators
    table_keywords = ["compare", "comparison", "table", "list", "types of", "specifications"]
    if any(kw in query_lower for kw in table_keywords):
        return "table"
    
    # Check for figure indicators
    figure_keywords = ["diagram", "figure", "circuit", "schematic", "waveform", "graph", "show me"]
    if any(kw in query_lower for kw in figure_keywords):
        return "figure"
    
    return "all"  # No specific intent


def intent_aware_search(query: str, top: int = 5) -> list[dict]:
    """
    Search with automatic intent detection.
    """
    intent = detect_intent(query)
    
    if intent == "table":
        filter_expr = "content_type eq 'table'"
        print(f"🎯 Detected intent: TABLE")
    elif intent == "figure":
        filter_expr = "content_type eq 'figure'"
        print(f"🎯 Detected intent: FIGURE")
    else:
        filter_expr = None
        print(f"🎯 Detected intent: GENERAL")
    
    return search_hybrid(query, top=top, filter=filter_expr)

# Test with different queries
print("\n" + "="*50)
results1 = intent_aware_search("Compare lap winding and wave winding")
print(f"Results: {[r['id'] for r in results1]}")

print("\n" + "="*50)
results2 = intent_aware_search("Show me the circuit diagram for a transformer")
print(f"Results: {[r['id'] for r in results2]}")

print("\n" + "="*50)
results3 = intent_aware_search("Explain Kirchhoff's current law")
print(f"Results: {[r['id'] for r in results3]}")

## Lab 4.3: RAG Pipeline Integration

This cell creates the **retrieval** portion of a complete RAG pipeline. Let's understand what it does:

### The RAG Query Flow

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         rag_query() Function                            │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Step 1: RETRIEVE                                                       │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  User Question → Semantic Search → Top K relevant chunks        │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                              ↓                                          │
│  Step 2: FORMAT CONTEXT                                                 │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Organize chunks by content type:                               │   │
│  │    • [Table 1]: ...    (for tables)                             │   │
│  │    • [Figure 1]: ...   (for figures)                            │   │
│  │    • [Text 1]: ...     (for text)                               │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                              ↓                                          │
│  Step 3: CREATE PROMPT                                                  │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  "Based on the following context, answer the question.          │   │
│  │   Context: [formatted chunks]                                   │   │
│  │   Question: [user question]                                     │   │
│  │   Answer:"                                                      │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### What the Function Returns

| Field | Description |
|-------|-------------|
| `question` | The original user question |
| `retrieved_chunks` | Number of chunks retrieved |
| `content_types` | List of types (e.g., `['text', 'table', 'text']`) |
| `context_length` | Character count of formatted context |
| `prompt` | Complete prompt ready for LLM |
| `chunks` | The actual chunk data for debugging |

### Why Separate Retrieval from Generation?

This separation enables:
- **Debugging**: Inspect which chunks were retrieved before sending to LLM
- **Token Management**: Check context length against model limits
- **Flexibility**: Swap LLMs without changing retrieval logic
- **Observability**: Log retrieval quality separately from generation quality

Lab 4.4 takes this prepared prompt and sends it to GPT-4.1 to generate the final answer.

In [ ]:
# Cell 4.3: Complete RAG query pipeline

def rag_query(
    question: str,
    top_k: int = 5,
    use_semantic: bool = True
) -> dict:
    """
    Complete RAG query: retrieve + format context for LLM.
    
    Args:
        question: User's question
        top_k: Number of chunks to retrieve
        use_semantic: Use semantic ranking
        
    Returns:
        Dict with retrieved context and metadata
    """
    # Step 1: Retrieve relevant chunks
    if use_semantic:
        search_result = search_semantic(question, top=top_k)
        chunks = search_result["results"]
    else:
        chunks = search_hybrid(question, top=top_k)
    
    # Step 2: Format context for LLM
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        # Format based on content type
        if chunk["content_type"] == "table":
            context_parts.append(f"[Table {i}]:\n{chunk['content']}")
        elif chunk["content_type"] == "figure":
            context_parts.append(f"[Figure {i} description]:\n{chunk['content']}")
        else:
            context_parts.append(f"[Text {i}]:\n{chunk['content']}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Step 3: Create prompt template
    prompt = f"""Based on the following context, answer the question.

Context:
{context}

Question: {question}

Answer:"""
    
    return {
        "question": question,
        "retrieved_chunks": len(chunks),
        "content_types": [c["content_type"] for c in chunks],
        "context_length": len(context),
        "prompt": prompt,
        "chunks": chunks
    }

# Test RAG query
question = "What are the different types of earthing and when should each be used?"
rag_result = rag_query(question, top_k=3)

print(f"🤖 RAG Query: '{question}'\n")
print(f"📊 Retrieved: {rag_result['retrieved_chunks']} chunks")
print(f"📁 Content types: {rag_result['content_types']}")
print(f"📏 Context length: {rag_result['context_length']} chars")
print(f"\n{'='*60}")
print("PROMPT (truncated):")
print(f"{'='*60}")
print(rag_result['prompt'][:1500] + "...")

## Lab 4.4: Generate Answer with GPT-4.1

In [ ]:
# Cell 4.4: Complete RAG with answer generation

def ask(question: str, top_k: int = 5) -> str:
    """
    Full RAG pipeline: retrieve → generate answer.
    
    Args:
        question: User's question
        top_k: Number of chunks to retrieve
        
    Returns:
        Generated answer
    """
    # Retrieve context
    rag_result = rag_query(question, top_k=top_k)
    
    # Generate answer
    chat_model = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-4.1")
    
    response = openai_client.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant for electrical engineering topics. Answer based on the provided context. If the context doesn't contain enough information, say so."
            },
            {
                "role": "user",
                "content": rag_result["prompt"]
            }
        ],
        temperature=0.3,
        max_tokens=500
    )
    
    return response.choices[0].message.content

# Test full RAG pipeline
question = "What is the difference between lap winding and wave winding in DC machines?"
print(f"❓ Question: {question}\n")
print("⏳ Generating answer...\n")

answer = ask(question)
print(f"💡 Answer:\n{answer}")

In [ ]:
# Cell 4.5: Test more questions

test_questions = [
    "What is the EMF equation for a DC generator?",
    "Explain the working principle of a transformer.",
    "What types of circuit breakers are used in low-tension switchgear?"
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"❓ {q}")
    print(f"{'='*60}")
    answer = ask(q, top_k=4)
    print(f"\n💡 {answer}")

---

# Part 5: Agentic Retrieval (Preview)

## What is Agentic Retrieval?

**Agentic Retrieval** is a new multi-query pipeline in Azure AI Search designed for complex questions posed by users or agents in chat and copilot apps. It's purpose-built for RAG patterns and agent-to-agent workflows.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                     Traditional RAG vs Agentic Retrieval                    │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  TRADITIONAL RAG:                                                           │
│  ┌─────────────────────────────────────────────────────────────────┐       │
│  │  User Question → Single Query → Search → Top K → LLM → Answer   │       │
│  └─────────────────────────────────────────────────────────────────┘       │
│                                                                             │
│  AGENTIC RETRIEVAL:                                                         │
│  ┌─────────────────────────────────────────────────────────────────┐       │
│  │  User Question + Chat History                                    │       │
│  │         ↓                                                        │       │
│  │  LLM Query Planning (decompose into focused subqueries)          │       │
│  │         ↓                                                        │       │
│  │  ┌─────────┐  ┌─────────┐  ┌─────────┐                          │       │
│  │  │Subquery1│  │Subquery2│  │Subquery3│  (parallel execution)    │       │
│  │  └────┬────┘  └────┬────┘  └────┬────┘                          │       │
│  │       ↓            ↓            ↓                                │       │
│  │  Semantic Rerank + Merge Results                                 │       │
│  │         ↓                                                        │       │
│  │  Unified Response (grounding data + references + activity plan)  │       │
│  └─────────────────────────────────────────────────────────────────┘       │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Key Capabilities

| Feature | Description |
|---------|-------------|
| **Query Decomposition** | LLM breaks complex questions into focused subqueries |
| **Parallel Execution** | All subqueries run simultaneously |
| **Chat Context** | Reads chat history for context-aware retrieval |
| **Spelling Correction** | Auto-corrects typos in queries |
| **Unified Response** | Merges results with references and execution metadata |

### Required Components

| Component | Service | Purpose |
|-----------|---------|---------|
| **LLM** | Azure OpenAI | Creates subqueries, answer synthesis |
| **Knowledge Base** | Azure AI Search | Orchestrates pipeline |
| **Knowledge Source** | Azure AI Search | Wraps search index |
| **Search Index** | Azure AI Search | Stores searchable content |
| **Semantic Ranker** | Azure AI Search | L2 reranking (required) |

> ⚠️ **Note**: Agentic Retrieval is in **public preview** and requires `azure-search-documents==11.7.0b2` or later.

## Lab 5.1: Install Preview SDK

Agentic Retrieval requires the beta version of the Azure Search SDK:

## Lab 5.0: Prerequisites - Upgrade Service & Configure RBAC

> ⚠️ **IMPORTANT**: Agentic Retrieval requires:
> - **Standard tier (S1) or higher** - Basic tier is NOT supported
> - **Role-based access control (RBAC)** enabled
> - **System-assigned managed identity** for your search service
> - **RBAC roles** assigned to your user account

The cell below will:
1. Check your current service tier
2. Upgrade to Standard tier if needed (⚠️ ~$250/month vs ~$75/month for Basic)
3. Enable role-based access
4. Create a system-assigned managed identity
5. Assign required RBAC roles to your user

**Run this cell only if you want to use Agentic Retrieval. Skip to the Summary section if you want to stay on Basic tier.**

In [ ]:
# Cell 5.0: Configure Search Service for Agentic Retrieval
# This cell configures your search service for agentic retrieval
# Prerequisites: Standard Premium Features must be enabled (see instructions below)

import subprocess
import json
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment
project_root = Path.cwd()
while project_root != project_root.parent:
    if (project_root / ".env").exists():
        load_dotenv(project_root / ".env")
        break
    project_root = project_root.parent

# Get search endpoint and extract service name
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# Extract service name from endpoint (e.g., https://my-search.search.windows.net -> my-search)
SEARCH_SERVICE_NAME = SEARCH_ENDPOINT.replace("https://", "").split(".")[0]

def run_az_command(cmd: str) -> dict:
    """Run Azure CLI command and return JSON result."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Command failed: {result.stderr[:200] if result.stderr else 'Unknown error'}")
        return None
    try:
        return json.loads(result.stdout) if result.stdout.strip() else {}
    except json.JSONDecodeError:
        return {"output": result.stdout}

print("🔍 Step 1: Checking current configuration...")
print(f"   Search service: {SEARCH_SERVICE_NAME}")

# Get current service info
service_info = run_az_command(
    f'az resource list --resource-type "Microsoft.Search/searchServices" '
    f'--query "[?name==\'{SEARCH_SERVICE_NAME}\']" -o json'
)

if not service_info:
    raise ValueError(f"❌ Could not find search service: {SEARCH_SERVICE_NAME}")

resource_group = service_info[0]["resourceGroup"]
print(f"   Resource group: {resource_group}")

# Get detailed service info
service_details = run_az_command(
    f'az search service show -n "{SEARCH_SERVICE_NAME}" -g "{resource_group}" -o json'
)

current_sku = service_details.get("sku", {}).get("name", "unknown")
semantic_search = service_details.get("semanticSearch", "disabled")
print(f"   SKU tier: {current_sku}")
print(f"   Semantic search: {semantic_search}")

# Check if Premium features are enabled
if semantic_search == "disabled" or semantic_search is None:
    print("\n" + "="*70)
    print("⚠️  PREMIUM FEATURES NOT ENABLED - ACTION REQUIRED")
    print("="*70)
    print("""
Agentic Retrieval requires 'Standard' Premium Features to be enabled.
This is a billing option, NOT the SKU tier.

📋 TO ENABLE PREMIUM FEATURES:
   1. Go to Azure Portal: https://portal.azure.com
   2. Navigate to your Search Service: {SEARCH_SERVICE_NAME}
   3. In the left menu, click: Settings → Premium features
   4. Under 'Availability', select: Standard ($0.00/month base + usage)
   5. Click 'Select plan' to confirm

💰 COST: 
   - Free tier: 1,000 semantic ranker requests/month
   - Standard tier: First 1,000 free, then $1.00/1,000 requests
   - Agentic retrieval: First 50M tokens free, then $0.022/1M tokens

After enabling Premium features, re-run this cell to continue setup.
""".format(SEARCH_SERVICE_NAME=SEARCH_SERVICE_NAME))
    print("="*70)
    raise ValueError("Please enable Premium features in Azure Portal, then re-run this cell.")

print(f"   ✅ Premium features enabled (semantic search: {semantic_search})")

# Step 2: Enable role-based access (with required aad-auth-failure-mode)
print("\n🔐 Step 2: Enabling role-based access control...")
rbac_result = run_az_command(
    f'az search service update -n "{SEARCH_SERVICE_NAME}" -g "{resource_group}" '
    f'--auth-options aadOrApiKey --aad-auth-failure-mode http401WithBearerChallenge -o json'
)
if rbac_result:
    print("   ✅ RBAC enabled (aadOrApiKey mode with bearer challenge)")

# Step 3: Enable system-assigned managed identity
print("\n🆔 Step 3: Enabling system-assigned managed identity...")
identity_result = run_az_command(
    f'az search service update -n "{SEARCH_SERVICE_NAME}" -g "{resource_group}" '
    f'--identity-type SystemAssigned -o json'
)
if identity_result:
    print("   ✅ System-assigned managed identity enabled")
    search_identity_principal = identity_result.get("identity", {}).get("principalId")
else:
    # Fetch it separately
    service_info_updated = run_az_command(
        f'az search service show -n "{SEARCH_SERVICE_NAME}" -g "{resource_group}" -o json'
    )
    search_identity_principal = service_info_updated.get("identity", {}).get("principalId")

print(f"   Identity Principal ID: {search_identity_principal}")

# Step 4: Get current user info
print("\n👤 Step 4: Getting current user info...")
user_info = run_az_command('az ad signed-in-user show -o json')
if user_info:
    user_id = user_info.get("id")
    user_name = user_info.get("userPrincipalName", user_info.get("displayName", "unknown"))
    print(f"   User: {user_name}")
else:
    # Fallback: get from account
    account_info = run_az_command('az account show -o json')
    user_id = account_info.get("user", {}).get("name")
    user_name = user_id
    print(f"   User: {user_name}")

# Step 5: Get subscription and resource IDs
subscription_info = run_az_command('az account show -o json')
subscription_id = subscription_info.get("id")
search_resource_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Search/searchServices/{SEARCH_SERVICE_NAME}"

# Step 6: Assign RBAC roles to user
print("\n🔑 Step 5: Assigning RBAC roles to user...")

roles = [
    ("Search Service Contributor", "7ca78c08-252a-4471-8644-bb5ff32d4ba0"),
    ("Search Index Data Contributor", "8ebe5a00-799e-43f5-93ac-243d3dce84a7"),
    ("Search Index Data Reader", "1407120a-92aa-4202-b7e9-c0e197c71c8f"),
]

for role_name, role_id in roles:
    print(f"   Assigning {role_name}...")
    assign_result = run_az_command(
        f'az role assignment create --assignee "{user_id}" '
        f'--role "{role_id}" --scope "{search_resource_id}" -o json 2>/dev/null'
    )
    if assign_result:
        print(f"   ✅ {role_name} assigned")
    else:
        print(f"   ⚠️  {role_name} - may already be assigned (this is OK)")

# Step 7: Grant Search Service identity access to Azure OpenAI
print("\n🤖 Step 6: Granting Search Service access to Azure OpenAI...")
print("   (Required for agentic retrieval query planning)")

# Find Azure OpenAI resource
openai_resources = run_az_command(
    f'az resource list --resource-type "Microsoft.CognitiveServices/accounts" '
    f'-g "{resource_group}" -o json'
)

if openai_resources and len(openai_resources) > 0:
    openai_resource_id = openai_resources[0]["id"]
    openai_name = openai_resources[0]["name"]
    print(f"   OpenAI resource: {openai_name}")
    
    # Assign Cognitive Services OpenAI User role to Search Service identity
    if search_identity_principal:
        print(f"   Assigning Cognitive Services OpenAI User to Search Service identity...")
        openai_role_result = run_az_command(
            f'az role assignment create --assignee "{search_identity_principal}" '
            f'--role "Cognitive Services OpenAI User" '
            f'--scope "{openai_resource_id}" -o json 2>/dev/null'
        )
        if openai_role_result:
            print(f"   ✅ Search Service can now access Azure OpenAI")
        else:
            print(f"   ⚠️  Role may already be assigned (this is OK)")
    else:
        print("   ⚠️  Could not get Search Service identity - assign manually in Azure Portal")
else:
    print(f"   ⚠️  No Azure OpenAI resource found in {resource_group}")
    print("   You may need to grant permissions manually")

# Final verification
print("\n" + "="*60)
print("📋 CONFIGURATION SUMMARY")
print("="*60)

# Re-fetch service info
final_info = run_az_command(
    f'az search service show -n "{SEARCH_SERVICE_NAME}" -g "{resource_group}" -o json'
)

print(f"   Service: {SEARCH_SERVICE_NAME}")
print(f"   SKU: {final_info.get('sku', {}).get('name', 'unknown')}")
print(f"   Semantic Search: {final_info.get('semanticSearch', 'unknown')}")
print(f"   Auth: {final_info.get('authOptions', 'unknown')}")
print(f"   Identity: {final_info.get('identity', {}).get('type', 'None')}")
print(f"   User: {user_name}")
print(f"\n   User Roles on Search:")
print(f"   - Search Service Contributor")
print(f"   - Search Index Data Contributor")
print(f"   - Search Index Data Reader")
print(f"\n   Search Service Identity Roles:")
print(f"   - Cognitive Services OpenAI User (on Azure OpenAI)")

print("\n✅ All prerequisites configured! You can now proceed with Lab 5.1")
print("   Wait 1-2 minutes for role assignments to propagate before running agentic retrieval.")

In [ ]:
# Cell 5.1: Install preview SDK for agentic retrieval
# The --pre flag installs the latest preview version with agentic retrieval support
import sys
!{sys.executable} -m pip install -q azure-search-documents --pre --force-reinstall

# Check installed version
import importlib.metadata
version = importlib.metadata.version('azure-search-documents')
print(f"✅ Installed azure-search-documents version: {version}")

# ⚠️ IMPORTANT: After running this cell, you MUST restart the kernel:
#    - Click "Restart" button in the notebook toolbar, OR
#    - Press Ctrl+Shift+P (Cmd+Shift+P on Mac) → "Jupyter: Restart Kernel"
#    - Then run Part 5 cells from the beginning (skip Parts 0-4 since data is already in the index)

print("\n⚠️  RESTART THE KERNEL NOW, then run cells starting from Cell 5.2")

## Lab 5.2: Create a Knowledge Source

A **Knowledge Source** is a reusable reference to source data (your search index). It specifies which fields to include in citation references.

In [ ]:
# Cell 5.2: Create a Knowledge Source
# Re-initialize required variables after kernel restart
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential

# Load environment variables
project_root = Path.cwd()
while project_root != project_root.parent:
    if (project_root / ".env").exists():
        load_dotenv(project_root / ".env")
        break
    project_root = project_root.parent

# Required configuration
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME", "rag-workshop-index")

# Try API key first (more reliable for admin operations), fall back to Entra ID
if SEARCH_API_KEY:
    credential = AzureKeyCredential(SEARCH_API_KEY)
    auth_method = "API Key"
else:
    credential = DefaultAzureCredential()
    auth_method = "Entra ID"

print(f"✅ Environment reloaded after kernel restart")
print(f"   - Search endpoint: {SEARCH_ENDPOINT[:50]}...")
print(f"   - Index name: {INDEX_NAME}")
print(f"   - Auth method: {auth_method}")

# Now import agentic retrieval classes
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    SearchIndexFieldReference,
)

# Knowledge source name
KNOWLEDGE_SOURCE_NAME = f"{INDEX_NAME}-knowledge-source"

# Create knowledge source pointing to our index
knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE_NAME,
    description="Knowledge source for RAG workshop electrical engineering documents",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        # Specify which fields to include in citation references
        # Exclude embedding field to avoid lengthy vectors in responses
        source_data_fields=[
            SearchIndexFieldReference(name="id"),
            SearchIndexFieldReference(name="content_type"),
            SearchIndexFieldReference(name="strategy"),
        ]
    ),
)

# Create index_client with the preview SDK
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

# Create or update the knowledge source
try:
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"\n✅ Knowledge source '{KNOWLEDGE_SOURCE_NAME}' created successfully")
    print(f"   - Target index: {INDEX_NAME}")
    print(f"   - Source fields: id, content_type, strategy")
except Exception as e:
    error_msg = str(e)
    print(f"\n❌ Error creating knowledge source: {error_msg[:200]}")
    
    if "Forbidden" in error_msg:
        print("\n⚠️  TROUBLESHOOTING - 'Forbidden' error usually means:")
        print("   1. Your search service tier may not support Agentic Retrieval")
        print("      - Requires Standard tier (S1) or higher")
        print("      - Basic tier does NOT support this feature")
        print("   2. Agentic Retrieval is in preview - ensure your region supports it")
        print("      - Supported regions: East US, West US 2, North Europe, West Europe")
        print("   3. If using Entra ID, you may need 'Search Service Contributor' role")
        print("\n   To check your service tier:")
        print("   - Go to Azure Portal → Your Search Service → Overview → Pricing tier")
    
    raise

## Lab 5.3: Create a Knowledge Base

A **Knowledge Base** orchestrates the agentic retrieval pipeline. It connects to:
- Your **LLM** (for query planning and answer synthesis)
- Your **Knowledge Sources** (for retrieval)

### Retrieval Reasoning Effort

| Effort Level | Description | Cost |
|--------------|-------------|------|
| **Minimal** | No LLM processing, direct search | Lowest |
| **Low** | Basic query reformulation | Low |
| **Medium** | Deeper search with follow-up iterations | Higher |

In [ ]:
# Cell 5.3: Create a Knowledge Base
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
    KnowledgeBaseAzureOpenAIModel,
    AzureOpenAIVectorizerParameters,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

# Knowledge base name
KNOWLEDGE_BASE_NAME = f"{INDEX_NAME}-knowledge-base"

# Configure the Azure OpenAI model for query planning
# Supported models: gpt-4o, gpt-4o-mini, gpt-4.1, gpt-4.1-nano, gpt-4.1-mini, gpt-5, gpt-5-nano, gpt-5-mini
# See: https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base#supported-models
#
# ⚠️ Agentic Retrieval requires models with Structured Outputs support
# gpt-4.1 (2025-04-14+) supports Structured Outputs and is the recommended model
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
GPT_MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_AGENTIC", "gpt-4.1")

print(f"📋 Configuring Knowledge Base with model: {GPT_MODEL}")

# Create model parameters (using same pattern as vectorizer parameters)
aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=GPT_MODEL,
    model_name=GPT_MODEL,
)

# Create knowledge base
knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)
    ],
    # Output mode: EXTRACTIVE_DATA returns raw content, ANSWER_SYNTHESIS uses LLM
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
    # Optional: Add instructions for answer synthesis (used when output_mode=ANSWER_SYNTHESIS)
    answer_instructions="Provide a concise, technical answer based on the retrieved documents. Include specific values and formulas when available.",
)

# Create or update the knowledge base
try:
    index_client.create_or_update_knowledge_base(knowledge_base)
    print(f"\n✅ Knowledge base '{KNOWLEDGE_BASE_NAME}' created successfully")
    print(f"   - Knowledge source: {KNOWLEDGE_SOURCE_NAME}")
    print(f"   - LLM model: {GPT_MODEL}")
    print(f"   - Output mode: Extractive Data")
except Exception as e:
    print(f"\n❌ Error creating knowledge base: {e}")
    raise

## Lab 5.4: Run Agentic Retrieval

Now let's test agentic retrieval with a complex, multi-part question that would be difficult for traditional single-query RAG.

In [ ]:
# Cell 5.4: Run agentic retrieval with the Knowledge Base Retrieval Client
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    SearchIndexKnowledgeSourceParams,
)

# Create the retrieval client
retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint=SEARCH_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=credential  # Uses Entra ID authentication
)

print(f"✅ Retrieval client created for knowledge base: {KNOWLEDGE_BASE_NAME}")

In [ ]:
# Cell 5.5: Test with a complex multi-part question
# This is a compound question that agentic retrieval will decompose into subqueries

complex_question = """
Compare the differences between lap winding and wave winding in DC machines. 
Also explain how the EMF equation relates to the type of winding used, 
and which winding type is better for high-current applications?
"""

# Build the retrieval request with conversation messages
retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=complex_question)]
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True
        )
    ],
    include_activity=True,
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort
)

print("📤 Sending complex query to agentic retrieval...")
print(f"   Query: {complex_question[:100]}...")
print("\n⏳ Processing (LLM will decompose into subqueries)...\n")

# Execute the retrieval
result = retrieval_client.retrieve(retrieval_request=retrieval_request)

print("✅ Retrieval complete!")

In [ ]:
# Cell 5.6: Inspect the agentic retrieval response
import json

# The response contains three parts:
# 1. Response - The retrieved content (or synthesized answer)
# 2. Activity - Execution details showing subqueries
# 3. References - Source documents with citations

print("="*80)
print("📊 AGENTIC RETRIEVAL RESPONSE")
print("="*80)

# Display response content
print("\n📝 RESPONSE:")
print("-"*40)
response_parts = []
for resp in result.response:
    for content in resp.content:
        if hasattr(content, 'text'):
            response_parts.append(content.text)

response_content = "\n\n".join(response_parts) if response_parts else "No response found"
# Truncate for display
print(response_content[:1500] + "..." if len(response_content) > 1500 else response_content)

# Display activity (subqueries executed)
print("\n\n🔍 ACTIVITY (Subqueries & Execution):")
print("-"*40)
if result.activity:
    activity_content = json.dumps([a.as_dict() for a in result.activity], indent=2)
    # Parse and display key info
    for activity in result.activity:
        activity_dict = activity.as_dict()
        activity_type = activity_dict.get('type', type(activity).__name__)
        print(f"\n• Activity Type: {activity_type}")
        
        if 'search_index_arguments' in activity_dict:
            search = activity_dict['search_index_arguments'].get('search', '')
            if search:
                print(f"  Subquery: {search[:100]}..." if len(str(search)) > 100 else f"  Subquery: {search}")
        
        if 'elapsed_ms' in activity_dict and activity_dict['elapsed_ms']:
            print(f"  Elapsed: {activity_dict['elapsed_ms']}ms")
        
        if 'input_tokens' in activity_dict:
            print(f"  Input tokens: {activity_dict['input_tokens']}")
        
        if 'output_tokens' in activity_dict:
            print(f"  Output tokens: {activity_dict['output_tokens']}")
else:
    print("No activity found")

# Display references
print("\n\n📚 REFERENCES (Source Documents):")
print("-"*40)
if result.references:
    for i, ref in enumerate(result.references[:5], 1):  # Show first 5
        ref_dict = ref.as_dict()
        print(f"\n{i}. Document ID: {ref_dict.get('doc_key', 'N/A')}")
        if 'reranker_score' in ref_dict and ref_dict['reranker_score']:
            print(f"   Reranker Score: {ref_dict['reranker_score']:.4f}")
    print(f"\n   (Showing 5 of {len(result.references)} references)")
else:
    print("No references found")

## Lab 5.5: Conversational Retrieval with Chat History

One of the key advantages of agentic retrieval is its ability to incorporate **chat history** for context-aware follow-up questions.

In [ ]:
# Cell 5.7: Conversational retrieval with chat history

# Simulate a multi-turn conversation using messages list
messages = [
    {"role": "user", "content": "What are the main components of a DC motor?"},
    {"role": "assistant", "content": "The main components of a DC motor include the armature, field winding, commutator, brushes, and yoke."},
    {"role": "user", "content": "How does the commutator work and what problems can occur with it?"}
]

# Build request with conversation history
conversational_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role=m["role"],
            content=[KnowledgeBaseMessageTextContent(text=m["content"])]
        ) for m in messages if m["role"] != "system"
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True
        )
    ],
    include_activity=True,
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort
)

print("💬 Conversational Query (with chat history):")
print("   Turn 1 (User): 'What are the main components of a DC motor?'")
print("   Turn 1 (Assistant): [Response about armature, field winding, etc.]")
print("   Turn 2 (User): 'How does the commutator work and what problems can occur with it?'")
print("\n⏳ Processing with context awareness...\n")

conversational_result = retrieval_client.retrieve(retrieval_request=conversational_request)

print("✅ Conversational retrieval complete!")
print(f"   Retrieved {len(conversational_result.references) if conversational_result.references else 0} relevant documents")
print(f"   Activities: {len(conversational_result.activity) if conversational_result.activity else 0}")

## Lab 5.6: Compare Traditional RAG vs Agentic Retrieval

Let's compare the same complex question using both approaches to see the difference:

In [ ]:
# Cell 5.8: Compare traditional RAG vs agentic retrieval
# Note: This cell is self-contained and works after kernel restart

import time
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from openai import AzureOpenAI
from azure.identity import get_bearer_token_provider

# Re-initialize clients if needed (after kernel restart)
if 'search_client' not in dir():
    search_client = SearchClient(
        endpoint=SEARCH_ENDPOINT,
        index_name=INDEX_NAME,
        credential=credential
    )

if 'openai_client' not in dir():
    token_provider = get_bearer_token_provider(
        credential if hasattr(credential, 'get_token') else DefaultAzureCredential(),
        "https://cognitiveservices.azure.com/.default"
    )
    openai_client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        azure_ad_token_provider=token_provider,
        api_version="2024-08-01-preview"
    )

EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_EMBEDDING", "text-embedding-3-large")

def get_embedding_simple(text: str) -> list[float]:
    """Generate embedding for text."""
    response = openai_client.embeddings.create(input=text, model=EMBEDDING_MODEL)
    return response.data[0].embedding

def search_hybrid_simple(query: str, top: int = 5) -> list[dict]:
    """Simple hybrid search for comparison."""
    query_embedding = get_embedding_simple(query)
    vector_query = VectorizedQuery(vector=query_embedding, k_nearest_neighbors=50, fields="embedding")
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        top=top,
        select=["id", "content", "content_type"]
    )
    return [{"id": r["id"], "content_type": r["content_type"]} for r in results]

# Comparison query
comparison_query = """
What are the differences between series and shunt DC motors? 
Which one has better starting torque and why? 
Also, what are their typical applications?
"""

print("="*80)
print("🔬 COMPARISON: Traditional RAG vs Agentic Retrieval")
print("="*80)
print(f"\nQuery: {comparison_query[:80]}...")

# Method 1: Traditional hybrid search
print("\n📊 METHOD 1: Traditional Hybrid Search")
print("-"*40)
start_time = time.time()
traditional_results = search_hybrid_simple(comparison_query, top=5)
traditional_time = (time.time() - start_time) * 1000

print(f"   Time: {traditional_time:.0f}ms")
print(f"   Documents: {len(traditional_results)}")
print(f"   Approach: Single query → Single search → Top K results")
print(f"   Document IDs: {[r['id'][:30] for r in traditional_results[:3]]}")

# Method 2: Agentic retrieval
print("\n📊 METHOD 2: Agentic Retrieval")
print("-"*40)
start_time = time.time()

agentic_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=comparison_query)]
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            always_query_source=True
        )
    ],
    include_activity=True,
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort
)
agentic_results = retrieval_client.retrieve(retrieval_request=agentic_request)
agentic_time = (time.time() - start_time) * 1000

# Count subqueries from activity
subquery_count = 0
if agentic_results.activity:
    for a in agentic_results.activity:
        activity_dict = a.as_dict()
        if activity_dict.get('type') == 'searchIndex':
            subquery_count += 1

print(f"   Time: {agentic_time:.0f}ms")
print(f"   Documents: {len(agentic_results.references) if agentic_results.references else 0}")
print(f"   Subqueries generated: {subquery_count}")
print(f"   Approach: Query decomposition → Parallel subqueries → Merged results")

# Show subqueries
if agentic_results.activity:
    print("\n   Generated Subqueries:")
    for activity in agentic_results.activity:
        activity_dict = activity.as_dict()
        if 'search_index_arguments' in activity_dict:
            search = activity_dict['search_index_arguments'].get('search', '')
            if search:
                subquery = search[:60] + "..." if len(str(search)) > 60 else search
                print(f"     • {subquery}")

print("\n" + "="*80)
print("💡 KEY DIFFERENCES:")
print("="*80)
print(f"""
| Aspect               | Traditional RAG          | Agentic Retrieval           |
|----------------------|--------------------------|------------------------------|
| Query handling       | Single query as-is       | Decomposed into subqueries   |
| Execution            | Sequential               | Parallel                     |
| Context awareness    | None                     | Uses chat history            |
| Complex questions    | May miss aspects         | Covers all parts             |
| Latency              | {traditional_time:.0f}ms{' '*(24-len(f'{traditional_time:.0f}ms'))}| {agentic_time:.0f}ms                        |
| Documents retrieved  | {len(traditional_results)}{' '*(24-len(str(len(traditional_results))))}| {len(agentic_results.references) if agentic_results.references else 0}                            |
| Cost                 | Lower                    | Higher (LLM tokens)          |
""")

## Lab 5.7: Cleanup Agentic Resources

Clean up the agentic retrieval resources (optional):

In [ ]:
# Cell 5.9: Cleanup agentic resources (OPTIONAL)
# Uncomment and run to delete knowledge base and knowledge source

# # Delete knowledge base first (depends on knowledge source)
# index_client.delete_knowledge_base(KNOWLEDGE_BASE_NAME)
# print(f"✅ Knowledge base '{KNOWLEDGE_BASE_NAME}' deleted")

# # Delete knowledge source
# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE_NAME)
# print(f"✅ Knowledge source '{KNOWLEDGE_SOURCE_NAME}' deleted")

print("💡 To delete agentic resources, uncomment the code above and run this cell.")

---

# Summary

## What We Learned

1. **Embeddings** capture semantic meaning in 3072-dimensional vectors
2. **Azure AI Search** provides a complete search platform with:
   - Text search (BM25)
   - Vector search (kNN)
   - Hybrid search (RRF fusion)
   - Semantic ranking (L2 reranker)
3. **Index design** matters: include content types for filtering
4. **Multi-retriever** patterns ensure balanced results across content types
5. **Full RAG pipeline**: Embed → Index → Retrieve → Generate
6. **Agentic Retrieval** (Preview) enables:
   - Query decomposition for complex questions
   - Parallel subquery execution
   - Chat history context awareness
   - Unified responses with citations

## Key Takeaways

| Component | Recommendation |
|-----------|----------------|
| Embedding Model | `text-embedding-3-large` (3072d) |
| Search Mode | Hybrid + Semantic for production |
| Index Design | Include `content_type` field |
| Retrieval | Multi-retriever for technical docs |
| Top-K | Start with 5, adjust based on context window |
| Complex Questions | Consider Agentic Retrieval |

## When to Use Agentic Retrieval

| Scenario | Use Traditional RAG | Use Agentic Retrieval |
|----------|--------------------|-----------------------|
| Simple factual queries | ✅ | ❌ (overkill) |
| Multi-part questions | ❌ | ✅ |
| Follow-up questions | ❌ | ✅ (context-aware) |
| Cost-sensitive apps | ✅ | ❌ (higher cost) |
| Agent-to-agent workflows | ❌ | ✅ |

---

## Next Steps

**Module 6: GraphRAG** - Cross-document reasoning with knowledge graphs

---

## Cleanup (Optional)

Run the cell below to delete the index if you want to start fresh:

In [ ]:
# Cell: Cleanup - Delete index (OPTIONAL)
# Uncomment and run if you want to delete the index

# index_client.delete_index(INDEX_NAME)
# print(f"✅ Index '{INDEX_NAME}' deleted")